In [1]:
import json
import os
from dotenv import load_dotenv
from openai import OpenAI

# ===============================
# LOAD ENV VARIABLES
# ===============================
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("❌ OPENAI_API_KEY not found in .env file")

# ===============================
# INIT CLIENT
# ===============================
client = OpenAI(api_key=api_key)

# ===============================
# LOAD INPUT JSON
# ===============================
with open("boq_input.json", "r") as f:
    data_list = json.load(f)

if not isinstance(data_list, list):
    data_list = [data_list]

print(f"📦 Total inputs: {len(data_list)}")

# ===============================
# CLEAN RESPONSE FUNCTION
# ===============================
def clean_output(text):
    text = text.strip()
    text = text.replace("```json", "").replace("```", "")
    return text.strip()

# ===============================
# BOQ GENERATION FUNCTION
# ===============================
def generate_boq(input_json):

  prompt = f"""
You are an advanced MEP BOQ (Bill of Quantities) estimation system.

Your task is to analyze structured input data and generate a BOQ by extracting elements, estimating quantities, and calculating costs in a consistent and controlled manner.

----------------------------------------
INPUT DATA:
{json.dumps(input_json, indent=2)}
----------------------------------------

🔍 INPUT UNDERSTANDING

- Carefully analyze the structure of the input
- Identify:
  • Available elements (e.g., ducts, pipes, cables, fittings, equipment)
  • Material rates
  • Any quantity-driving parameters (e.g., floor area, counts, lengths)

- Allow equivalent field interpretation:
  • e.g., floor_area_sqft → floor_area

IMPORTANT:
- Use ONLY the data present in the input
- Do NOT assume completely missing information
- Do NOT introduce new elements

----------------------------------------
🧠 ELEMENT PROCESSING

- Process ONLY elements explicitly present in the input
- Maintain logical grouping of elements
- Do NOT expand categories into new components

🚫 EQUIPMENT RULE:
- Include equipment ONLY if explicitly defined in the input
- Otherwise, skip it entirely

----------------------------------------
📐 QUANTITY ESTIMATION (CONTROLLED & GENERIC)

- If a driving parameter (e.g., area, count, length) is available:
  → derive quantities using simple proportional relationships

- If quantities are already provided:
  → use them directly

- If data is insufficient:
  → perform minimal estimation without assumptions

----------------------------------------
🔢 QUANTITY RULES

- All quantities MUST be integers
- Decimal values are NOT allowed

Rounding:
- ≥ 0.5 → round up
- < 0.5 → round down

Apply to all count-based elements

----------------------------------------
💰 COST CALCULATION

- Use ONLY material_rates from input
- Map elements directly to available rates

- total_cost = quantity × unit_rate

CRITICAL:
- If valid rate exists → MUST calculate cost
- If rate is missing or invalid → set:
  • unit_rate = 0
  • total_cost = 0

- All cost values MUST be ≥ 0

----------------------------------------
⚠️ VALIDATION & ERROR HANDLING

- Validate consistency between elements and rates

- If invalid or unsupported data is detected:
  → skip or neutralize affected item (set cost = 0)
  → continue processing remaining elements

- Do NOT stop execution due to partial errors

----------------------------------------
📊 OUTPUT GENERATION

- Generate structured JSON dynamically based on input
- Include ONLY elements that exist in input

For each element include:
  • quantity / length
  • unit_rate (if available)
  • total_cost

Also include:
  • grand_total_cost (sum of all valid costs)

----------------------------------------
🧾 OUTPUT RULES

- Output MUST be valid JSON
- All numeric values MUST be ≥ 0
- Do NOT include negative values
- Do NOT return empty output if valid data exists

----------------------------------------
📊 FINAL OUTPUT ORDER

1. Elements (grouped logically)
2. "grand_total_cost"
3. "remarks"
4. "confidence_score"

----------------------------------------
🧾 REMARKS RULE

- Single sentence only
- Maximum 12 words
- Summarize estimation or highlight issues

----------------------------------------
📈 CONFIDENCE SCORE

- Float between 0 and 1
- Based on data completeness:
  • Complete → high (0.8–0.95)
  • Partial → medium (0.5–0.8)
  • Limited → low (<0.5)

----------------------------------------
🚫 STRICT CONSTRAINTS

- ❌ Do NOT hallucinate
- ❌ Do NOT introduce new elements
- ❌ Do NOT infer missing categories
- ❌ Do NOT include equipment unless explicitly present
- ❌ Do NOT skip calculation when valid data exists

----------------------------------------
🎯 FINAL INSTRUCTION

Think like a cost engineer.

Adapt to the input structure.
Extract → Validate → Estimate → Calculate → Output.

Return ONLY valid JSON.
"""
  response = client.chat.completions.create(
          model="gpt-5-mini",   # ✅ stable + cost efficient
          messages=[{"role": "user", "content": prompt}]
      )

  raw = response.choices[0].message.content
  cleaned = clean_output(raw)

  return json.loads(cleaned)

# ===============================
# MAIN LOOP
# ===============================
results = []

for i, input_json in enumerate(data_list):

    print(f"\n🔹 Processing input {i+1}")

    try:
        output = generate_boq(input_json)

        results.append({
            "input": input_json,
            "output": output
        })

        # PRINT
        print("\n Input:")
        print(json.dumps(input_json, indent=2))

        print("\n Output:")
        print(json.dumps(output, indent=2))

        print("\n" + "="*60)

    except Exception as e:
        print("❌ Error:", str(e))

# ===============================
# SAVE OUTPUT
# ===============================
with open("output_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n✅ BOQ generation completed and saved!")

📦 Total inputs: 1

🔹 Processing input 1

 Input:
{
  "project_type": "HVAC",
  "drawing_file": "floor_plan_hvac.pdf",
  "floor_area_sqft": 2500,
  "units": "metric",
  "elements_to_extract": [
    "ducts",
    "diffusers",
    "fittings"
  ],
  "material_rates": {
    "duct_15x7_per_meter": 25,
    "diffuser_unit": 15,
    "elbow_unit": 10
  }
}

 Output:
{
  "elements": {
    "ducts": {
      "length_m": 116,
      "unit_rate": 25,
      "total_cost": 2900
    },
    "diffusers": {
      "quantity": 12,
      "unit_rate": 15,
      "total_cost": 180
    },
    "fittings": {
      "elbow": {
        "quantity": 39,
        "unit_rate": 10,
        "total_cost": 390
      }
    }
  },
  "grand_total_cost": 3470,
  "remarks": "Estimated quantities based on floor area; limited input provided.",
  "confidence_score": 0.65
}


✅ BOQ generation completed and saved!
